<a href="https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My lane as an ML task (type)

**Lane:** SEO Content Prioritization

**Task Type:** Ranking (Scoring)

The goal is to rank content pages based on how urgently they should be refreshed. Instead of predicting only "refresh" or "don't refresh", the model assigns each page a priority score so editors can update the highest-impact pages first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or proxy

The starter dataset does not contain future content performance, so I use a proxy target.

I define pages with a negative trend (`trend_pct < -10`) as pages that need refreshing.

This label is created using a rule from the available data rather than an observed future outcome.

In [ ]:
!git clone https://github.com/krihna7/flyrank-ml-internship.git
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success Metric

**Success Metric:** Precision@50

Precision@50 measures how many of the top 50 pages recommended by the model actually need refreshing.

This metric is appropriate because editors can only refresh a limited number of pages. A higher Precision@50 means the model is placing the most important pages near the top of the ranking, making it more useful for decision-making.

In [ ]:
# Create the proxy target (if it doesn't already exist)
df["target_refresh"] = (df["trend_pct"] < -10).astype(int)

# Show the distribution of the target
print("Target distribution:")
print(df["target_refresh"].value_counts())

print("\nPercentage distribution:")
print((df["target_refresh"].value_counts(normalize=True) * 100).round(2))

# Preview the target column
df[["trend_pct", "target_refresh"]].head()

Target distribution:
target_refresh
1    18248
0    11752
Name: count, dtype: int64

Percentage distribution:
target_refresh
1    60.83
0    39.17
Name: proportion, dtype: float64


,trend_pct,target_refresh
0,-41.4,1
1,-57.7,1
2,-60.9,1
3,-13.8,1
4,-34.7,1


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## The unit of analysis, as a real dataframe

One row represents one content page for one client, summarised using SEO performance data from the previous 90 days.

Each row contains features such as impressions, clicks, CTR, average position, engagement rate, content age and trend information. These features describe the performance of a single content page and are used by the model to determine its refresh priority.

In [ ]:
# Section 4: The unit of analysis as a real dataframe

# Display the dataset dimensions
print("Dataset shape:", df.shape)

# Show the number of rows and columns
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

# Display the first five rows of the dataset
print("\nFirst 5 rows:")
display(df.head())

# Display key columns that represent one unit of analysis
print("\nSample of the unit of analysis:")
display(df[[
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_pct"
]].head())

Dataset shape: (30000, 45)
Number of rows: 30000
Number of columns: 45

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,target_refresh
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1



Sample of the unit of analysis:


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,trend_pct
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,-41.4
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,-60.9
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Why ML beats a fixed rule here

A fixed rule such as "refresh every page with a traffic decline greater than 10%" is too simple because it only considers one feature.

Content performance depends on many factors working together, including impressions, clicks, CTR, average search position, engagement rate, AI traffic, content age, and trend direction. The relationship between these features is complex and varies across pages.

A machine learning model can learn these patterns from the data and rank pages based on their overall likelihood of benefiting from a content refresh. This provides better decision support than relying on a single threshold or if-statement.

In [ ]:
# Show some of the features available for making predictions

features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "ai_traffic_pct",
    "content_age_days",
    "trend_pct"
]

print("Features that could be used by an ML model:")
display(df[features].head())

Features that could be used by an ML model:


,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,ai_traffic_pct,content_age_days,trend_pct
0,3803,29,0.76,10.6,5.88,0.0,187,-41.4
1,15320,7,0.05,20.3,0.00,0.0,445,-57.7
2,12581,11,0.09,36.5,0.00,0.0,141,-60.9
3,11751,58,0.49,6.2,1.28,0.0,463,-13.8
4,19140,24,0.13,44.0,0.00,0.0,263,-34.7


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.